# 폰 영상 한 편으로 방의 3D 점 구름 + 카메라 경로 복원

모델: `facebook/map-anything-apache` (T4 무료 등급 기준, fp16)

순서: GPU 확인 → 설정값 → **모델 설치/다운로드(4.91GB, 먼저 시작)** → 영상 업로드 → 프레임 추출 → 복원 → 후처리 → 저장/다운로드 → 3D 가우시안 스플랫 PLY 저장

## 0. 그래픽 장치 확인

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경에서 T4 GPU를 선택하세요."

props = torch.cuda.get_device_properties(0)
print(f"GPU 이름: {props.name}")
print(f"전체 메모리: {props.total_memory / 1e9:.2f} GB")
print(f"계산 능력(compute capability): {props.major}.{props.minor}")

## 1. 설정값

In [ ]:
VIDEO_PATH = "/content/IMG_3352.MOV"
MODEL_NAME = "facebook/map-anything-apache"

N_FRAMES = 24          # 영상에서 고르게 뽑을 장 수
MAX_LONG_SIDE = 1024   # 뽑은 사진의 긴 변을 이 값 이하로 줄인다

LOW_PCT, HIGH_PCT = 0.5, 99.5   # 각 축에서 버릴 위/아래 퍼센트
CAM_SPREAD_RATIO_MIN = 0.1      # 카메라 이동 범위 / 장면 대각선, 이보다 작으면 경고

K = 1.4        # 가우시안 알갱이 반지름 배율, 손으로 조정 가능
N_KEEP = 0     # 0이면 전부, 숫자를 넣으면 그만큼만 고르게 뽑아 스플랫으로 저장

OUT_DIR = "/content/output"
FRAME_DIR = "/content/frames"
import os
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FRAME_DIR, exist_ok=True)

## 2. 모델 설치 및 다운로드 (영상 업로드보다 먼저)

가중치가 4.91GB라서 내려받는 동안 아래 셀들을 미리 읽어 둘 수 있도록 이 단계를 영상 업로드보다 앞에 둔다.

`pip install` 직후에는 방금 생긴 site-packages 경로가 현재 파이썬 프로세스의 `sys.path`/경로 탐색 캐시에 반영되어 있지 않아 `import`가 실패한다. 런타임을 재시작하지 않고 넘어가기 위해, 새로 깔린 자리를 `sys.path`에 직접 끼워 넣고 캐시를 무효화한다.

In [ ]:
import sys, site, importlib, subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/facebookresearch/map-anything.git"],
    check=True,
)

# 새로 설치된 site-packages 자리를 검색 목록 맨 앞에 끼워 넣고,
# 예전에 캐시된 디렉터리 목록을 무효화해서 런타임 재시작 없이 넘어간다.
importlib.reload(site)
for p in site.getsitepackages() + [site.getusersitepackages()]:
    if p and p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

from mapanything.models import MapAnything
print("map-anything import 성공")

In [ ]:
model = MapAnything.from_pretrained(MODEL_NAME)
model = model.to("cuda").eval()
print("모델 로드 완료:", MODEL_NAME)

## 3. 영상 업로드

In [ ]:
import os
from google.colab import files

if not os.path.exists(VIDEO_PATH):
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
    os.rename(fname, VIDEO_PATH)

print("영상 준비 완료:", VIDEO_PATH, f"({os.path.getsize(VIDEO_PATH) / 1e6:.1f} MB)")

## 4. 프레임 24장 고르게 뽑기 + 긴 변 줄이기

뽑은 사진은 나중에 함께 내려받을 수 있도록 바로 파일로도 저장해 둔다.

In [ ]:
import cv2
import numpy as np

def extract_frames(video_path, n_frames, max_long_side, save_dir):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs = np.linspace(0, total - 1, n_frames).round().astype(int)

    frames = []
    for order, i in enumerate(idxs):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame_bgr = cap.read()
        if not ok:
            raise RuntimeError(f"{i}번째 프레임을 읽지 못했습니다.")

        frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        h, w = frame.shape[:2]
        scale = max_long_side / max(h, w)
        if scale < 1.0:
            frame = cv2.resize(frame, (round(w * scale), round(h * scale)), interpolation=cv2.INTER_AREA)

        cv2.imwrite(os.path.join(save_dir, f"frame_{order:02d}.jpg"),
                    cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
        frames.append(frame)

    cap.release()
    return frames

frames = extract_frames(VIDEO_PATH, N_FRAMES, MAX_LONG_SIDE, FRAME_DIR)
print(f"{len(frames)}장 추출, 크기 예시: {frames[0].shape}")

In [ ]:
import matplotlib.pyplot as plt

check_idxs = np.linspace(0, len(frames) - 1, 6).round().astype(int)
fig, axes = plt.subplots(1, len(check_idxs), figsize=(18, 3))
for ax, i in zip(axes, check_idxs):
    ax.imshow(frames[i])
    ax.set_title(f"frame {i}")
    ax.axis("off")
plt.show()

## 5. 복원 실행 (fp16)

bf16은 T4에서 메모리가 터지므로 fp16으로 지정한다.

> `mapanything`의 정확한 출력 키 이름은 버전에 따라 달라질 수 있다. 아래 셀이 `predictions[0].keys()`를 찍어 주니, 다음 셀에서 꺼내 쓰는 키 이름(`pts3d`, `camera_poses` 등)이 다르면 여기서 확인해서 맞춰 주면 된다.

In [ ]:
import time
from mapanything.utils.image import load_images

views = load_images(FRAME_DIR)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()

with torch.no_grad():
    predictions = model.infer(
        views,
        memory_efficient_inference=False,
        use_amp=True,
        amp_dtype="fp16",   # bf16은 T4에서 메모리가 터진다
        apply_mask=True,
        mask_edges=True,
    )

torch.cuda.synchronize()
elapsed = time.time() - t0
peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9

print(f"복원 시간: {elapsed:.1f}초")
print(f"최대 GPU 메모리: {peak_mem_gb:.2f} GB")
print("출력 키 목록(필요하면 아래 셀에서 이름을 맞춰 쓸 것):", list(predictions[0].keys()))

## 6. 점/색/카메라 위치 꺼내기

In [ ]:
all_points, all_colors, cam_positions = [], [], []

for pred, frame in zip(predictions, frames):
    pts = np.asarray(pred["pts3d"]).reshape(-1, 3)
    cols = (frame.reshape(-1, 3).astype(np.float32)) / 255.0

    all_points.append(pts)
    all_colors.append(cols)

    pose = np.asarray(pred["camera_poses"])  # 4x4, world 좌표계 기준 카메라 위치로 가정
    cam_positions.append(pose[:3, 3])

points = np.concatenate(all_points, axis=0)
colors = np.concatenate(all_colors, axis=0)
cam_positions = np.stack(cam_positions, axis=0)

print(f"점 {len(points)}개, 카메라 {len(cam_positions)}개")

## 7. 이상치 제거 (각 축 상하 0.5%)

창밖이나 거울에 비친 점이 멀리 날아가 장면을 한없이 키우는 것을 막는다.

In [ ]:
def remove_axis_outliers(points, colors, low_pct, high_pct):
    mask = np.ones(len(points), dtype=bool)
    for axis in range(3):
        lo, hi = np.percentile(points[:, axis], [low_pct, high_pct])
        mask &= (points[:, axis] >= lo) & (points[:, axis] <= hi)
    return points[mask], colors[mask]

points, colors = remove_axis_outliers(points, colors, LOW_PCT, HIGH_PCT)
print(f"이상치 제거 후 남은 점: {len(points)}개")

## 8. 카메라 이동 범위 점검

카메라 위치들이 퍼진 범위를 장면 대각선으로 나눈 값이 0.1보다 작으면, 촬영 범위가 좁아 복원이 부정확할 수 있다는 신호.

In [ ]:
scene_min, scene_max = points.min(axis=0), points.max(axis=0)
scene_diagonal = float(np.linalg.norm(scene_max - scene_min))

cam_min, cam_max = cam_positions.min(axis=0), cam_positions.max(axis=0)
cam_spread = float(np.linalg.norm(cam_max - cam_min))

ratio = cam_spread / scene_diagonal
print(f"카메라 이동 범위 / 장면 대각선 = {ratio:.3f}")

if ratio < CAM_SPREAD_RATIO_MIN:
    print("카메라가 거의 한 자리에서만 찍힌 것 같습니다. 더 걸어 다니며 다시 찍어 주세요.")

## 9. 점 구름 + 카메라 위치 + 뽑은 사진 저장/다운로드

In [ ]:
import zipfile

npz_path = os.path.join(OUT_DIR, "room_reconstruction.npz")
np.savez(npz_path, points=points, colors=colors, camera_positions=cam_positions)

zip_path = os.path.join(OUT_DIR, "room_reconstruction.zip")
with zipfile.ZipFile(zip_path, "w") as zf:
    zf.write(npz_path, arcname="room_reconstruction.npz")
    for fname in sorted(os.listdir(FRAME_DIR)):
        zf.write(os.path.join(FRAME_DIR, fname), arcname=f"frames/{fname}")

print(f"저장 완료: {zip_path} ({os.path.getsize(zip_path) / 1e6:.2f} MB)")
files.download(zip_path)

## 10. 3D 가우시안 스플랫 .ply 로 저장

- 좌표는 모델이 준 값 그대로 쓰되 Y, Z 부호를 뒤집는다 (모델은 Y가 아래, 웹 3D 라이브러리는 Y가 위).
- 색은 0차 구면 조화 계수(`f_dc`)로만 넣고, 1차 이상(`f_rest_*`)은 헤더에서 아예 뺀다.
- 알갱이 반지름 = `K × 장면의 가장 긴 변 ÷ √N`, 저장 시 로그를 취해 `scale_*`에 넣는다.
- 불투명도는 0.9의 시그모이드 역함수 값을 `opacity`에 넣는다.
- 회전은 항등 사원함수 `(1,0,0,0)`, 법선은 0.
- INRIA 3D Gaussian Splatting 포맷과 같은 속성 이름/순서의 이진 `.ply`.

In [ ]:
C0 = 0.28209479177387814  # 구면 조화 0차 계수 정규화 상수

def rgb_to_sh_dc(colors_0_1):
    return (colors_0_1 - 0.5) / C0

def inverse_sigmoid(x):
    return float(np.log(x / (1 - x)))

def save_gaussian_ply(path, points, colors, K, N_KEEP):
    t0 = time.time()

    pos = points.astype(np.float32).copy()
    pos[:, 1] *= -1  # Y 부호 뒤집기
    pos[:, 2] *= -1  # Z 부호 뒤집기

    longest_side = float((pos.max(axis=0) - pos.min(axis=0)).max())

    if N_KEEP and 0 < N_KEEP < len(pos):
        idx = np.linspace(0, len(pos) - 1, N_KEEP).round().astype(int)
        pos, cols = pos[idx], colors[idx]
    else:
        cols = colors

    n = len(pos)
    radius = K * longest_side / np.sqrt(n)
    log_scale = float(np.log(radius))  # 3DGS는 지수 활성화 전 로그값을 저장한다

    f_dc = rgb_to_sh_dc(cols).astype(np.float32)
    opacity = np.full((n, 1), inverse_sigmoid(0.9), dtype=np.float32)
    scale = np.full((n, 3), log_scale, dtype=np.float32)
    rot = np.zeros((n, 4), dtype=np.float32)
    rot[:, 0] = 1.0  # 회전 없음: (w, x, y, z) = (1, 0, 0, 0)
    normals = np.zeros((n, 3), dtype=np.float32)

    attrs = np.concatenate([pos, normals, f_dc, opacity, scale, rot], axis=1).astype("<f4")

    names = (["x", "y", "z", "nx", "ny", "nz", "f_dc_0", "f_dc_1", "f_dc_2", "opacity"]
             + ["scale_0", "scale_1", "scale_2"]
             + ["rot_0", "rot_1", "rot_2", "rot_3"])

    header = "ply\nformat binary_little_endian 1.0\n"
    header += f"element vertex {n}\n"
    for name in names:
        header += f"property float {name}\n"
    header += "end_header\n"

    with open(path, "wb") as f:
        f.write(header.encode("ascii"))
        f.write(attrs.tobytes())

    elapsed = time.time() - t0
    size_mb = os.path.getsize(path) / 1e6

    print(f"알갱이 개수: {n}")
    print(f"반지름: {radius:.5f}")
    print(f"장면의 가장 긴 변: {longest_side:.3f}")
    print(f"파일 크기: {size_mb:.2f} MB")
    print(f"쓰는 데 걸린 시간: {elapsed:.2f}초")

ply_path = os.path.join(OUT_DIR, "room_gaussians.ply")
save_gaussian_ply(ply_path, points, colors, K=K, N_KEEP=N_KEEP)
files.download(ply_path)